In [2]:
import os
import random
import h5py

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision.models as models
from torchinfo import summary
from sklearn.metrics import r2_score
import plotly.express as px


from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split


import torchvision.transforms.functional as TF

import seaborn as sns
import matplotlib.pyplot as plt


sns.set_theme(style="ticks", palette="pastel", rc={"lines.linewidth": 2.5})

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

generator = torch.Generator().manual_seed(SEED)

# Set the device to use for training
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Data

In [3]:
folder_path = "models/"

# Create the folder path and checkpoints directory if they don't exist
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

if not os.path.exists(os.path.join(folder_path, "checkpoints")):
    os.makedirs(os.path.join(folder_path, "checkpoints"))

- Reading files

In [4]:
# modify so indices is the index of the dataframe
def read_hdf5_to_dataframe_with_index(h5_path="unified_parallel.h5"):
    with h5py.File(h5_path, "r") as f:
        viirs_start = f["viirs_start"][:]
        viirs_end = f["viirs_end"][:]
        rgb = f["rgb"][:]
        figures = f["figures"][:]
        indices = f["indices"][:]
        iso3 = f["iso3"][:]
        types = f["type"][:]

    # Decode bytes to strings for iso3
    iso3_decoded = [x.decode("utf-8") if isinstance(x, bytes) else x for x in iso3]
    types_decoded = [x.decode("utf-8") if isinstance(x, bytes) else x for x in types]

    # Create a DataFrame with indices as the index
    df = pd.DataFrame(
        {
            "viirs_start": list(viirs_start),
            "viirs_end": list(viirs_end),
            "rgb": list(rgb),
            "figures": figures,
            "iso3": iso3_decoded,
            "type": types_decoded,
        },
        index=indices,
    )

    df.sort_index(inplace=True)  # Ensure indices are sorted

    return df

In [5]:
path = "../src/data/processed/disaster.h5"
df_disaster = read_hdf5_to_dataframe_with_index(path)

path_idu = "../src/data/processed/testing.h5"
df_idu = read_hdf5_to_dataframe_with_index(path_idu)

# load csv iso3 embeddings
iso3_embeddings = pd.read_csv("../src/data/processed/embeddings_mapped.csv", index_col=0)

# combine the two dataframes
df = pd.concat([df_disaster, df_idu], ignore_index=True)

del df_disaster
del df_idu

In [6]:
len(df)

16896

In [7]:
iso3_embeddings.head()

,emb_0,emb_1,emb_2,emb_3
iso3_mapped,,,,
AND,-2.155526,-0.099045,1.090775,0.091681
ARE,-2.354902,-0.086402,1.196469,0.499288
AFG,2.381075,-0.013857,-0.791427,0.159821
ATG,-0.600651,-0.696483,-1.296734,1.598701
ALB,0.138654,-0.234058,-0.686938,-0.162994


In [8]:
import numpy as np
import plotly.express as px
import pycountry # You may need to: pip install pycountry

def plot_iso3_distribution_nature(df):
    # 1. Aggregate data
    iso3_counts = df["iso3"].value_counts().reset_index()
    iso3_counts.columns = ["iso3", "count"]
    iso3_counts["log_count"] = np.log1p(iso3_counts["count"])
    
    # 2. Map ISO3 to Country Names
    def get_country_name(code):
        try:
            return pycountry.countries.get(alpha_3=code).name
        except:
            return code # Fallback to code if not found

    iso3_counts["country_name"] = iso3_counts["iso3"].apply(get_country_name)
    
    # Get top 10 for the bar chart
    top_10 = iso3_counts.nlargest(10, "count").sort_values("count")

    # --- Figure 1: Choropleth (Keep ISO3 for locations) ---
    fig_map = px.choropleth(
        iso3_counts,
        locations="iso3",
        color="log_count",
        hover_name="country_name", # Changed to country name for better UX
        hover_data={"count": True, "log_count": False, "iso3": False},
        color_continuous_scale="Greys",
    )

    fig_map.update_geos(
        projection_type="natural earth",
        showframe=False,
        showcoastlines=False,
        showland=True,
        landcolor="rgb(240,240,240)",
        bgcolor="white"
    )

    fig_map.update_layout(
        template="simple_white",
        height=560,
        width=1000,
        margin=dict(l=10, r=10, t=60, b=10),
        font=dict(family="Arial", size=11, color="black"),
        title=dict(
            text="Climate-related disaster concentration by country<br><sup>Global distribution</sup>",
            x=0.01,
            xanchor="left",
            font=dict(size=14)
        ),
        coloraxis_colorbar=dict(
            title="Log events",
            thickness=10,
            len=0.6,
            outlinewidth=0
        ),
    )

    fig_map.write_image("distribution_map.pdf", engine="kaleido")
    fig_map.show()

    # --- Figure 2: Bar chart (Use country_name for Y-axis) ---
    fig_bar = px.bar(
        top_10,
        y="country_name", # Changed from "iso3"
        x="count",
        orientation="h",
        color_discrete_sequence=["#4D4D4D"],
    )

    fig_bar.update_traces(
        marker_line_width=0,
        opacity=0.9
    )

    fig_bar.update_layout(
        template="simple_white",
        height=480,
        width=700,
        margin=dict(l=50, r=10, t=60, b=20), # Increased left margin for long names
        font=dict(family="Arial", size=11, color="black"),
        title=dict(
            text="Top countries by climate-related disasters",
            x=0.01,
            xanchor="left",
            font=dict(size=14)
        ),
        xaxis=dict(
            title="Events",
            showgrid=True,
            gridcolor="rgba(0,0,0,0.05)",
            zeroline=False
        ),
        yaxis=dict(
            title="",
            showgrid=False,
            autorange="reversed" # Optional: keeps highest at the top
        ),
        showlegend=False
    )

    fig_bar.write_image("top_10_bar.pdf", engine="kaleido")
    fig_bar.show()

    # --- Summary ---
    share = top_10["count"].sum() / iso3_counts["count"].sum() * 100
    print(f"Top 10 countries account for {share:.1f}% of events")
    print(", ".join(top_10["country_name"].tolist()))

plot_iso3_distribution_nature(df)

/var/folders/ys/5gn1z90x7c1_khtjpp5kg6580000gn/T/ipykernel_28599/3615676146.py:62: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig_map.write_image("distribution_map.pdf", engine="kaleido")


/var/folders/ys/5gn1z90x7c1_khtjpp5kg6580000gn/T/ipykernel_28599/3615676146.py:105: DeprecationWarning: 
Support for the 'engine' argument is deprecated and will be removed after September 2025.
Kaleido will be the only supported engine at that time.

  fig_bar.write_image("top_10_bar.pdf", engine="kaleido")


Top 10 countries account for 54.5% of events
Sri Lanka, Malaysia, Philippines, United States, Burundi, India, Indonesia, Brazil, Somalia, Peru


# Simple baseline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

# =========================================================
# CONFIG
# =========================================================
RANDOM_STATE = 42
TEST_SIZE = 0.1
BOOTSTRAP_ITERS = 1000

# =========================================================
# 1. FEATURE ENGINEERING MODULE
# =========================================================
def get_image_stats(image_array):
    """Extracts 8 statistical moments from an image array."""
    if image_array is None or len(image_array) == 0:
        return np.zeros(8)
    try:
        img = np.array(image_array)
        if img.size == 0: return np.zeros(8)
        flat = img.flatten()
        return np.array([
            np.mean(flat), np.std(flat), np.min(flat), np.max(flat),
            np.percentile(flat, 25), np.percentile(flat, 50),
            np.percentile(flat, 75), np.ptp(flat)
        ])
    except:
        return np.zeros(8)

def process_image_columns(df, columns_map):
    """Processes multiple image columns into a single feature DataFrame."""
    all_stats = []
    for col, prefix in columns_map.items():
        stats = df[col].apply(get_image_stats)
        stat_df = pd.DataFrame(np.array(stats.tolist()), 
                               columns=[f"{prefix}_{i}" for i in range(8)])
        all_stats.append(stat_df)
    return pd.concat(all_stats, axis=1)

def prepare_features(df, iso3_embeddings):
    """Merges embeddings, creates dummies, and builds image features."""
    embedding_df = iso3_embeddings.reset_index().rename(
        columns={iso3_embeddings.index.name or "index": "iso3"}
    )
    merged = df.merge(embedding_df, on="iso3", how="inner")
    
    type_dummies = pd.get_dummies(merged["type"], prefix="type")
    
    img_map = {"viirs_start": "vs", "viirs_end": "ve", "rgb": "rgb"}
    img_features = process_image_columns(merged, img_map)
    
    embed_cols = [c for c in embedding_df.columns if c != "iso3"]
    
    X_full_df = pd.concat([merged[embed_cols], type_dummies, img_features], axis=1)
    y = merged["figures"].to_numpy(np.float32)
    
    return X_full_df, y, embed_cols

# =========================================================
# 2. MODELING MODULE
# =========================================================
def get_xgb_regressor():
    """Returns a pre-configured XGBoost model."""
    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=5000,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=100
    )

def run_bootstrap_analysis(y_true, y_pred, n_iterations=BOOTSTRAP_ITERS):
    """Calculates distribution of metrics via bootstrapping."""
    stats = {"R2": [], "MAE": [], "RMSE": []}
    rng = np.random.default_rng(RANDOM_STATE)
    for _ in range(n_iterations):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        y_t, y_p = y_true[idx], y_pred[idx]
        stats["R2"].append(r2_score(y_t, y_p))
        stats["MAE"].append(mean_absolute_error(y_t, y_p))
        stats["RMSE"].append(np.sqrt(mean_squared_error(y_t, y_p)))
    return pd.DataFrame(stats)

# =========================================================
# 3. VISUALIZATION MODULE
# =========================================================
def format_feature_labels(feature_names):
    """Converts internal codes to human-readable labels."""
    stat_map = {0:"mean", 1:"std", 2:"min", 3:"max", 4:"25th", 5:"50th", 6:"75th", 7:"ptp"}
    labels = []
    for f in feature_names:
        if f.startswith("type_"): labels.append(f"Type: {f[5:]}")
        elif any(f.startswith(p) for p in ["vs_","ve_","rgb_"]):
            p, i = f.split("_")
            p_map = {"vs":"VIIRS Start", "ve":"VIIRS End", "rgb":"RGB"}
            labels.append(f"{p_map[p]} {stat_map[int(i)]}")
        else: labels.append(f"Embed: {f}")
    return labels

def export_performance_pdf(y_test, pred_dict, boot_dict, imp_df, output_path="Analysis_Report.pdf"):
    """Generates the 4-panel diagnostic plot with error bars and saves to PDF."""
    plt.rcParams.update({"font.family": "Arial", "font.size": 9})
    fig = plt.figure(figsize=(16, 5), dpi=300)
    gs = GridSpec(1, 4, width_ratios=[1, 1, 0.8, 0.9])

    colors = ["#4C72B0", "#55A868"]
    
    # Panels 1 & 2: Scatters
    for i, (name, y_pred) in enumerate(pred_dict.items()):
        ax = fig.add_subplot(gs[0, i])
        ax.scatter(y_test, y_pred, s=20, alpha=0.4, color=colors[i], edgecolors='none')
        ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "--k", lw=1)
        ax.set_title(name, weight='bold')
        ax.set_xlabel("Observed")
        ax.set_ylabel("Predicted")
        ax.grid(True, linestyle="--", alpha=0.2)

    # Panel 3: R2 Stability
    ax_boot = fig.add_subplot(gs[0, 2])
    v = ax_boot.violinplot([boot_dict["A"]["R2"], boot_dict["B"]["R2"]], showmedians=True)
    for i, pc in enumerate(v['bodies']):
        pc.set_facecolor(colors[i]); pc.set_edgecolor('black'); pc.set_alpha(0.6)
    ax_boot.set_xticks([1, 2]); ax_boot.set_xticklabels(["Model A", "Model B"])
    ax_boot.set_title("Statistical Stability ($R^2$)", weight='bold')

    # Panel 4: Feature Importance with Error Bars
    ax_imp = fig.add_subplot(gs[0, 3])
    top_imp = imp_df.head(12).iloc[::-1]
    
    ax_imp.barh(top_imp["display_name"], 
                top_imp["importance_mean"], 
                xerr=top_imp["importance_std"], 
                color="#C44E52",
                alpha=0.8,
                error_kw={'capsize': 2, 'elinewidth': 0.8})
    
    ax_imp.set_title("Feature Importance", weight='bold')
    ax_imp.set_xlabel("Drop in $R^2$")
    ax_imp.grid(True, axis='x', linestyle="--", alpha=0.2)

    plt.tight_layout()
    plt.savefig(output_path, format='pdf', bbox_inches='tight')
    # plt.close()
    print(f"Report successfully saved to: {output_path}")

# =========================================================
# MAIN EXECUTION
# =========================================================
if __name__ == "__main__":
    # Ensure variables 'df' and 'iso3_embeddings' are defined in your environment
    
    # 1. Feature Engineering
    X_full, y, embed_cols = prepare_features(df, iso3_embeddings)
    y_log = np.log1p(y)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_full, y_log, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )

    # 2. Training
    model_a = get_xgb_regressor()
    model_a.fit(X_train[embed_cols], y_train, 
                eval_set=[(X_test[embed_cols], y_test)], verbose=False)
    
    model_b = get_xgb_regressor()
    model_b.fit(X_train, y_train, 
                eval_set=[(X_test, y_test)], verbose=False)

    # 3. Evaluation & Bootstrapping
    preds_a = model_a.predict(X_test[embed_cols])
    preds_b = model_b.predict(X_test)
    
    boot_a = run_bootstrap_analysis(y_test, preds_a)
    boot_b = run_bootstrap_analysis(y_test, preds_b)

    # 4. Importance Calculation with Std Dev
    imp = permutation_importance(model_b, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE)
    
    imp_df = pd.DataFrame({
        "feature": X_full.columns,
        "importance_mean": imp.importances_mean,
        "importance_std": imp.importances_std
    }).sort_values("importance_mean", ascending=False)
    
    imp_df["display_name"] = format_feature_labels(imp_df["feature"])

    # 5. PDF Export
    export_performance_pdf(
        y_test = y_test,
        pred_dict = {"Model A: Embeddings": preds_a, "Model B: Combined": preds_b},
        boot_dict = {"A": boot_a, "B": boot_b},
        imp_df = imp_df,
        output_path = "Model_Performance_Analysis.pdf"
    )

Report successfully saved to: Model_Performance_Analysis.pdf
